# Cybersecurity Incident Analyzer — Fase 2

## Prompt Engineering e Saídas Controladas

//TODO validar outputs

---

### Objetivo

Este notebook demonstra **técnicas de prompt engineering** aplicadas à triagem automatizada de incidentes de cibersegurança para times SOC (Security Operations Center).

Três técnicas são implementadas, comparadas e discutidas:

| # | Técnica | Ideia central |
|---|---------|---------------|
| 1 | **Role + Context + Task + Output Format (RCTOF)** | Prompt totalmente especificado com role priming e esquema JSON explícito |
| 2 | **Guided Chain of Thought (GCoT)** | O prompt guia um raciocínio em etapas antes da saída final |
| 3 | **Meta Prompting** | O LLM gera o prompt ideal; esse prompt conduz a análise final |

As três técnicas usam o **mesmo relato de incidente** e miram o **mesmo esquema de saída em JSON**.

> **Dependência de dados:** Requer `transformers`, `torch`, `accelerate`, `pydantic` e `pandas`. Este notebook lê o dataset compartilhado em `data/incidents.json` para manter a mesma base entre as fases. Não há RAG nem embeddings.


---
## 1. Setup e Imports

In [1]:
import json
import re
import textwrap
from enum import Enum
from pathlib import Path

import pandas as pd
import torch
from pydantic import BaseModel, Field, field_validator
from transformers import AutoModelForCausalLM, AutoTokenizer


In [6]:
!git clone https://github.com/gabriel-q7/LLM-cybersecurity-analyzer

Cloning into 'LLM-cybersecurity-analyzer'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (20/20), done.
Receiving objects: 100% (29/29), 49.50 KiB | 9.90 MiB/s, done.
remote: Total 29 (delta 7), reused 24 (delta 6), pack-reused 0 (from 0)
Resolving deltas: 100% (7/7), done.


In [2]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 768

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
elif torch.cuda.is_available():
    MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MODEL_DEVICE = next(model.parameters()).device

print(f"Model  : {MODEL_ID}")
print(f"Device : {MODEL_DEVICE}")
print(f"DType  : {MODEL_DTYPE}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model  : Qwen/Qwen2.5-1.5B-Instruct
Device : cuda:0
DType  : torch.bfloat16


---
## 2. Definição do Incidente

Os incidentes são carregados do dataset compartilhado em `data/incidents.json`. Um item é selecionado com `INCIDENT_INDEX` e reutilizado nas três técnicas para permitir comparação direta.

> **Dataset base:** os 10 incidentes fictícios usados em todo o projeto. A seleção inicial abaixo usa o incidente de índice `0`.


In [4]:
DATA_PATH = Path("data/incidents.json")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found at data/incidents.json. Run the repository setup cell first."
    )

def load_incidents(data_path: Path) -> list[dict]:
    return json.loads(data_path.read_text(encoding="utf-8"))


incidents = load_incidents(DATA_PATH)
INCIDENT_INDEX = 0
selected_incident = incidents[INCIDENT_INDEX]
INCIDENT_TEXT = selected_incident["text"]

print(f"Loaded {len(incidents)} incidents from {DATA_PATH.name}")
print(f"Using incident #{selected_incident['id']}")
print(f"Incident length: {len(INCIDENT_TEXT)} characters")
print()
print("─" * 70)
for line in textwrap.wrap(INCIDENT_TEXT, width=70):
    print(line)
print("─" * 70)


FileNotFoundError: Shared incidents dataset not found. Clone the repository or upload data/incidents.json to the runtime.

---
## 3. Esquema de Saída (Pydantic v2)

As três técnicas precisam produzir saídas compatíveis com este esquema fixo.

O Pydantic v2 valida:
- `severity` pertence a `low | medium | high | critical` (Enum, rejeitando strings inválidas)
- `confidence` é um float `0.0 ≤ x ≤ 1.0`
- `recommended_actions` é uma lista não vazia de strings não vazias
- `summary` tem no mínimo 10 caracteres

In [ ]:
class SeverityLevel(str, Enum):
    low = "low"
    medium = "medium"
    high = "high"
    critical = "critical"


class IncidentAnalysis(BaseModel):
    incident_type: str = Field(..., description="Type of cybersecurity incident")
    severity: SeverityLevel = Field(..., description="Severity: low/medium/high/critical")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Confidence score 0.0-1.0")
    summary: str = Field(..., min_length=10, description="2-3 sentence incident summary")
    recommended_actions: list[str] = Field(
        ..., min_length=1, description="Prioritized SOC response actions"
    )

    @field_validator("confidence")
    @classmethod
    def round_confidence(cls, v: float) -> float:
        return round(v, 3)

    @field_validator("recommended_actions")
    @classmethod
    def actions_not_empty(cls, v: list[str]) -> list[str]:
        if not all(isinstance(a, str) and a.strip() for a in v):
            raise ValueError("All actions must be non-empty strings")
        return v


print("Schema: IncidentAnalysis")
print()
schema = IncidentAnalysis.model_json_schema()
print(json.dumps(schema, indent=2))

Schema: IncidentAnalysis

{
  "$defs": {
    "SeverityLevel": {
      "enum": [
        "low",
        "medium",
        "high",
        "critical"
      ],
      "title": "SeverityLevel",
      "type": "string"
    }
  },
  "properties": {
    "incident_type": {
      "description": "Type of cybersecurity incident",
      "title": "Incident Type",
      "type": "string"
    },
    "severity": {
      "$ref": "#/$defs/SeverityLevel",
      "description": "Severity: low/medium/high/critical"
    },
    "confidence": {
      "description": "Confidence score 0.0-1.0",
      "maximum": 1.0,
      "minimum": 0.0,
      "title": "Confidence",
      "type": "number"
    },
    "summary": {
      "description": "2-3 sentence incident summary",
      "minLength": 10,
      "title": "Summary",
      "type": "string"
    },
    "recommended_actions": {
      "description": "Prioritized SOC response actions",
      "items": {
        "type": "string"
      },
      "minItems": 1,
      "title": "R

---
## 4. Utilitários de Geração, Parsing e Validação

Pipeline completo: `prompt de chat → geração com modelo open source → parse de JSON → validação com Pydantic`

### Tratamento de erros

| Modo de falha | Tratamento |
|---|---|
| JSON envolto em markdown fences | Remoção via regex |
| `json.loads()` falha | `ValueError` com contexto |
| Validação do Pydantic falha | `ValidationError` com detalhe por campo |

In [ ]:
def generate_response_text(system_prompt, user_prompt, max_new_tokens=MAX_NEW_TOKENS):
    # Monta a conversa no formato esperado pelo modelo e gera apenas a continuação.
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(prompt_text, return_tensors="pt")
    model_inputs = {k: v.to(MODEL_DEVICE) for k, v in model_inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_tokens = model_inputs["input_ids"].shape[-1]
    completion_tokens = generated_ids.shape[-1] - prompt_tokens
    completion_ids = generated_ids[0][prompt_tokens:]
    completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

    return completion_text, prompt_tokens, completion_tokens


def parse_json_response(text):
    # Extrai e interpreta um objeto JSON a partir da saída textual bruta.
    text = text.strip()

    # 1. Tentativa direta
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2. Remoção de cercas de markdown
    for pattern in [r"```json\s*(.*?)\s*```", r"```\s*(.*?)\s*```"]:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1).strip())
            except json.JSONDecodeError:
                continue

    # 3. Busca gulosa pelo primeiro bloco completo { ... }
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    raise ValueError(f"No valid JSON found in response:\n{text[:500]}")


def validate_analysis(data):
    # Valida o dicionário contra o schema IncidentAnalysis.
    return IncidentAnalysis(**data)


def run_analysis(system_prompt, user_prompt, label="Analysis", max_new_tokens=MAX_NEW_TOKENS):
    # Executa a geração, interpreta o JSON e valida a resposta final.
    print(f"[{label}] Sending request...")

    text, input_tokens, output_tokens = generate_response_text(
        system_prompt,
        user_prompt,
        max_new_tokens=max_new_tokens,
    )

    raw_data = None
    analysis = None
    parse_ok = False
    validation_ok = False
    error = None

    try:
        raw_data = parse_json_response(text)
        parse_ok = True
    except ValueError as e:
        error = f"ParseError: {e}"

    if parse_ok:
        try:
            analysis = validate_analysis(raw_data)
            validation_ok = True
        except Exception as e:
            error = f"ValidationError: {e}"

    status = "OK" if validation_ok else f"FAILED — {error}"
    print(f"[{label}] Status  : {status}")
    print(f"[{label}] Tokens  : {input_tokens} in / {output_tokens} out")

    return {
        "label": label,
        "text": text,
        "raw_data": raw_data,
        "analysis": analysis,
        "parse_ok": parse_ok,
        "validation_ok": validation_ok,
        "error": error,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }


def display_result(result):
    # Exibe de forma legível uma resposta validada.
    sep = "=" * 65
    print(sep)
    print(f"  {result['label']}")
    print(sep)

    if not result["validation_ok"]:
        print(f"  ERROR: {result['error']}")
        if result["text"]:
            print(f"\n  Raw text (first 800 chars):\n{result['text'][:800]}")
        return

    analysis = result["analysis"]
    print(f"  Incident Type  : {analysis.incident_type}")
    print(f"  Severity       : {analysis.severity.value.upper()}")
    print(f"  Confidence     : {analysis.confidence:.3f}")
    print(f"  Summary        : {analysis.summary}")
    print(f"  Actions ({len(analysis.recommended_actions)}):")
    for i, action in enumerate(analysis.recommended_actions, 1):
        print(f"    {i}. {action}")
    print(f"  Tokens         : {result['input_tokens']} in / {result['output_tokens']} out")


print("Utilities loaded.")

Utilities loaded.


---
## 5. Técnica 1 — Role + Context + Task + Output Format (RCTOF)

### Racional de desenho

O padrão **RCTOF** organiza o prompt em quatro elementos explícitos:

| Elemento | Conteúdo | Propósito |
|----------|----------|------------|
| **Role** | Analista sênior de cibersegurança | Priming de domínio, alinhando o comportamento do modelo |
| **Context** | Cenário de triagem SOC | Enquadramento situacional, reduzindo ambiguidades |
| **Task** | Classificar, avaliar, resumir e recomendar | Escopo explícito da instrução |
| **Output Format** | Esquema JSON exato dentro do prompt | Restrição estrutural da saída |

### Comportamento esperado
O modelo produz JSON diretamente, sem explicações extras. Há apenas uma geração, com baixo overhead.

In [ ]:
T1_SYSTEM = (
    "You are a senior cybersecurity analyst specializing in incident response and threat intelligence. "
    "You have extensive experience with SOC operations, incident classification, and triage procedures "
    "across large enterprise environments."
)

_JSON_SCHEMA = (
    '{\n'
    '  "incident_type": "<string: attack category>",\n'
    '  "severity": "<one of: low | medium | high | critical>",\n'
    '  "confidence": <float: 0.0 to 1.0>,\n'
    '  "summary": "<string: 2-3 sentence description of what happened>",\n'
    '  "recommended_actions": [\n'
    '    "<string: first prioritized action>",\n'
    '    "<string: second prioritized action>"\n'
    '  ]\n'
    '}'
)

T1_USER = (
    "## Context\n"
    "Your organization's SOC has received a cybersecurity incident report requiring immediate triage.\n\n"
    "## Task\n"
    "Analyze the incident report below. Identify the attack type, assess severity, estimate your "
    "confidence level, write a concise summary, and list prioritized response actions.\n\n"
    "## Incident Report\n"
    + INCIDENT_TEXT
    + "\n\n"
    "## Output Format\n"
    "Respond with ONLY a valid JSON object — no explanation, no markdown fences, no preamble.\n"
    "Use exactly this schema:\n"
    + _JSON_SCHEMA
)

print(f"System prompt : {len(T1_SYSTEM)} chars")
print(f"User prompt   : {len(T1_USER)} chars")
print()
print("─" * 65)
print(T1_USER[:650])
print("... [truncated]")

System prompt : 233 chars
User prompt   : 1597 chars

─────────────────────────────────────────────────────────────────
## Context
Your organization's SOC has received a cybersecurity incident report requiring immediate triage.

## Task
Analyze the incident report below. Identify the attack type, assess severity, estimate your confidence level, write a concise summary, and list prioritized response actions.

## Incident Report
On March 3rd, at 03:12 AM, the file server hosting the HR department's shared drive became unresponsive. Upon investigation, system administrators discovered that thousands of files had been encrypted and renamed with a '.lockedcorp' extension. A ransom note demanding 5 Bitcoin was found in every affected directory, threatening to leak e
... [truncated]


In [ ]:
result_t1 = run_analysis(T1_SYSTEM, T1_USER, label="T1: RCTOF")
print()
display_result(result_t1)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[T1: RCTOF] Sending request...
[T1: RCTOF] Status  : OK
[T1: RCTOF] Tokens  : 382 in / 102 out

  T1: RCTOF
  Incident Type  : Data Encryption Ransomware
  Severity       : HIGH
  Confidence     : 0.900
  Summary        : A ransomware attack encrypted thousands of files on an HR department's shared drive, demanding payment via a .onion domain.
  Actions (2):
    1. Contain the spread of the ransomware by isolating the infected network segment.
    2. Secure backups and restore them as soon as possible.
  Tokens         : 382 in / 102 out


---
## 6. Técnica 2 — Guided Chain of Thought (GCoT)

### Racional de desenho

**Chain of Thought** orienta o modelo a organizar o raciocínio em etapas intermediárias antes de chegar à conclusão. Aqui, essas etapas são **explicitamente guiadas** e cada uma aponta para um campo específico da saída.

| Etapa | Alvo do raciocínio | Campo de saída |
|------|---------------------|----------------|
| 1 | Identificação do vetor de ataque | ajuda `incident_type` |
| 2 | Classificação do incidente | `incident_type` |
| 3 | Avaliação de severidade (tríade CIA) | `severity` |
| 4 | Calibração de confiança baseada em evidências | `confidence` |
| 5 | Resumo do incidente | `summary` |
| 6 | Ações priorizadas de resposta | `recommended_actions` |

### Comportamento esperado
Mesmo sem expor o raciocínio intermediário, a estrutura guiada tende a melhorar a calibração e a consistência da saída, ao custo de prompts mais longos.

In [ ]:
T2_SYSTEM = (
    "You are a senior cybersecurity analyst with deep expertise in threat analysis and incident response. "
    "You approach every incident with systematic, evidence-based reasoning — examining attack vectors, "
    "blast radius, and response urgency before reaching conclusions."
)

_COT_STEPS = (
    "**Step 1 — Identify the attack vector:**\n"
    "What mechanism was used? Look for: delivery method, exploitation technique, initial access path.\n\n"
    "**Step 2 — Classify the incident type:**\n"
    "Based on the vector and observed behavior, assign an incident type. Consider: Ransomware, "
    "Phishing, Malware, DDoS, Insider Threat, Credential Theft, Brute Force, Supply Chain, or other.\n\n"
    "**Step 3 — Assess severity (low / medium / high / critical):**\n"
    "Evaluate business impact across:\n"
    "- Confidentiality: Was sensitive data exposed or at risk?\n"
    "- Integrity: Were systems or data modified?\n"
    "- Availability: Were services disrupted?\n"
    "- Scope: How many users/systems affected?\n"
    "- Urgency: Is the threat ongoing?\n\n"
    "**Step 4 — Estimate confidence (0.0–1.0):**\n"
    "How certain are you given available evidence?\n"
    "- High (>0.85): clear indicators, multiple corroborating signals\n"
    "- Medium (0.65–0.85): some ambiguity or missing evidence\n"
    "- Low (<0.65): significant uncertainty or contradictory signals\n\n"
    "**Step 5 — Write a concise incident summary (2-3 sentences):**\n"
    "Cover: what happened, who/what was affected, key indicators observed.\n\n"
    "**Step 6 — List prioritized response actions:**\n"
    "Specific, actionable steps for the SOC team, ordered by priority.\n\n"
)

_FINAL_JSON = (
    '{\n'
    '  "incident_type": "<string>",\n'
    '  "severity": "<low|medium|high|critical>",\n'
    '  "confidence": <float 0.0-1.0>,\n'
    '  "summary": "<string>",\n'
    '  "recommended_actions": ["<string>", ...]\n'
    '}'
)

T2_USER = (
    "Analyze this cybersecurity incident using the structured reasoning steps below.\n"
    "Work through each step carefully before producing the final JSON output.\n\n"
    "**Incident Report:**\n"
    + INCIDENT_TEXT
    + "\n\n---\n\n"
    + _COT_STEPS
    + "---\n\n"
    "After completing all steps, output ONLY a valid JSON object:\n"
    + _FINAL_JSON
)

print(f"System prompt : {len(T2_SYSTEM)} chars")
print(f"User prompt   : {len(T2_USER)} chars")
print()
print("─" * 65)
print(T2_USER[:650])
print("... [truncated]")

System prompt : 262 chars
User prompt   : 2460 chars

─────────────────────────────────────────────────────────────────
Analyze this cybersecurity incident using the structured reasoning steps below.
Work through each step carefully before producing the final JSON output.

**Incident Report:**
On March 3rd, at 03:12 AM, the file server hosting the HR department's shared drive became unresponsive. Upon investigation, system administrators discovered that thousands of files had been encrypted and renamed with a '.lockedcorp' extension. A ransom note demanding 5 Bitcoin was found in every affected directory, threatening to leak employee personal data if payment was not made within 72 hours. The ransom note contained a contact email on a .onion domain. Initial for
... [truncated]


In [ ]:
result_t2 = run_analysis(T2_SYSTEM, T2_USER, label="T2: Guided CoT")
print()
display_result(result_t2)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[T2: Guided CoT] Sending request...
[T2: Guided CoT] Status  : FAILED — ValidationError: 1 validation error for IncidentAnalysis
severity
  Input should be 'low', 'medium', 'high' or 'critical' [type=enum, input_value='High', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/enum
[T2: Guided CoT] Tokens  : 602 in / 150 out

  T2: Guided CoT
  ERROR: ValidationError: 1 validation error for IncidentAnalysis
severity
  Input should be 'low', 'medium', 'high' or 'critical' [type=enum, input_value='High', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/enum

  Raw text (first 800 chars):
```json
{
  "incident_type": "Ransomware",
  "severity": "High",
  "confidence": 0.9,
  "summary": "A ransomware attack has occurred on the HR department's shared drive, encrypting over 47,000 files with a '.lockedcorp' extension. The attackers demanded 5 Bitcoin via a .onion domain. Initial access was gained through a compromised RDP s

---
## 7. Técnica 3 — Meta Prompting

### Racional de desenho

**Meta prompting** usa o próprio LLM para gerar ou melhorar um prompt e depois aplica esse prompt à tarefa real. Essa abordagem em duas fases pode revelar estruturas de prompt que um engenheiro humano talvez não desenhasse diretamente.

**Fase 1 — Geração do prompt:** pedir ao mesmo modelo open source que desenhe o melhor prompt possível para análise de incidentes.

**Fase 2 — Aplicação do prompt:** usar o prompt gerado (mais o texto do incidente) para a análise efetiva.

### Fluxo de iteração

```
[v1: naive] → [meta: generate optimal prompt] → [v2: generated] → [run analysis]
```

### Comportamento esperado
O prompt gerado pode incorporar escolhas estruturais inesperadas, enquadramentos novos e restrições adicionais, potencialmente melhorando a qualidade final.

In [ ]:
META_GEN_SYSTEM = (
    "You are an expert prompt engineer specializing in designing AI prompts for high-stakes "
    "security operations systems. You understand how to elicit accurate, calibrated, and "
    "structured outputs from large language models."
)

_SCHEMA_SPEC = (
    '{ "incident_type": str, "severity": "low|medium|high|critical", '
    '"confidence": float 0-1, "summary": str, "recommended_actions": list[str] }'
)

META_GEN_USER = (
    "Design the single most effective prompt for a cybersecurity incident analysis AI assistant.\n\n"
    "The prompt must be optimized to:\n"
    "1. Elicit accurate classification of cybersecurity incident types across diverse attack categories\n"
    "2. Produce well-calibrated severity ratings (low/medium/high/critical) based on evidence\n"
    "3. Generate a confidence score (0.0-1.0) reflecting evidence quality and completeness\n"
    "4. Produce a concise, factual 2-3 sentence incident summary\n"
    "5. Generate specific, prioritized SOC response actions\n"
    "6. Enforce strict JSON output with this schema (no deviation):\n"
    "   " + _SCHEMA_SPEC + "\n\n"
    "The prompt must be robust across: Ransomware, Phishing, DDoS, Insider Threat, "
    "Credential Theft, Malware, and Supply Chain attacks.\n\n"
    "Return ONLY the final prompt text. No meta-commentary, no explanation, no section headers."
)

print("Phase 1: Generating optimal analysis prompt via meta-prompting...")
print()

GENERATED_PROMPT, meta_input_tokens, meta_output_tokens = generate_response_text(
    META_GEN_SYSTEM,
    META_GEN_USER,
    max_new_tokens=512,
)

print("Meta-prompt generated.")
print(f"Tokens: {meta_input_tokens} in / {meta_output_tokens} out")
print()
print("─" * 65)
print("GENERATED PROMPT:")
print("─" * 65)
print(GENERATED_PROMPT)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Phase 1: Generating optimal analysis prompt via meta-prompting...

Meta-prompt generated.
Tokens: 260 in / 64 out

─────────────────────────────────────────────────────────────────
GENERATED PROMPT:
─────────────────────────────────────────────────────────────────
```json
{
  "incident_type": "<type>",
  "severity": "<severity>",
  "confidence": "<confidence_score>",
  "summary": "<brief_summary>",
  "recommended_actions": [
    "<action_1>",
    "<action_2>",
    "<action_3>"
  ]
}
```


In [ ]:
# Comentário em português: a análise final continua usando prompts em inglês.
T3_SYSTEM = "You are a senior cybersecurity analyst."

T3_USER = (
    GENERATED_PROMPT
    + "\n\n---\n\nIncident report to analyze:\n\n"
    + INCIDENT_TEXT
)

print("Phase 2: Running analysis with the generated prompt...")
print()
result_t3 = run_analysis(T3_SYSTEM, T3_USER, label="T3: Meta Prompting")
print()
display_result(result_t3)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Phase 2: Running analysis with the generated prompt...

[T3: Meta Prompting] Sending request...
[T3: Meta Prompting] Status  : FAILED — ParseError: No valid JSON found in response:
Based on the provided incident report, here is an analysis of the situation:

### Incident Type:
The incident type appears to be a **Data Encryption Ransomware Attack**.

### Severity:
This attack has a high severity level due to the widespread nature of the encryption and the potential impact on sensitive employee information. The attackers have threatened to release personal data unless payment is made, which could cause significant distress and loss of trust among employees.

### Confidence S
[T3: Meta Prompting] Tokens  : 253 in / 501 out

  T3: Meta Prompting
  ERROR: ParseError: No valid JSON found in response:
Based on the provided incident report, here is an analysis of the situation:

### Incident Type:
The incident type appears to be a **Data Encryption Ransomware Attack**.

### Severity:
This atta

---
## 8. Demonstração de Iteração de Prompts

As três técnicas representam uma evolução desde um prompt ingênuo até desenhos progressivamente mais sofisticados.

```
v1 (Base) ──► v2 (RCTOF) ──► v3 (Guided CoT) ──► v4 (Meta-gerado)
    ↓              ↓                 ↓                    ↓
Sem estrutura  Papel explícito   Raciocínio guiado   Estrutura desenhada pelo LLM
Sem schema     + schema JSON     passo a passo       automaticamente
```

In [ ]:
V1_PROMPT = (
    "Analyze this cybersecurity incident and output a JSON object with: "
    "incident_type, severity, confidence, summary, recommended_actions.\n\n"
    + INCIDENT_TEXT
)

prompts = [
    ("v1 — Baseline (no structure)", V1_PROMPT),
    ("v2 — RCTOF (Technique 1)", T1_USER),
    ("v3 — Guided CoT (Technique 2)", T2_USER),
    ("v4 — Meta-generated (Technique 3)", T3_USER),
]

for label, prompt in prompts:
    print(f"{'─' * 65}")
    print(f"  {label}")
    print(f"  Length: {len(prompt)} chars")
    print()
    preview = " ".join(prompt[:280].split())
    print(f"  Preview: {preview}...")
    print()

─────────────────────────────────────────────────────────────────
  v1 — Baseline (no structure)
  Length: 959 chars

  Preview: Analyze this cybersecurity incident and output a JSON object with: incident_type, severity, confidence, summary, recommended_actions. On March 3rd, at 03:12 AM, the file server hosting the HR department's shared drive became unresponsive. Upon investigation, system administrator...

─────────────────────────────────────────────────────────────────
  v2 — RCTOF (Technique 1)
  Length: 1597 chars

  Preview: ## Context Your organization's SOC has received a cybersecurity incident report requiring immediate triage. ## Task Analyze the incident report below. Identify the attack type, assess severity, estimate your confidence level, write a concise summary, and list prioritized respons...

─────────────────────────────────────────────────────────────────
  v3 — Guided CoT (Technique 2)
  Length: 2460 chars

  Preview: Analyze this cybersecurity incident using the 

---
## 9. Tabela Comparativa

Resultados lado a lado das três técnicas sobre o mesmo incidente.

In [ ]:
all_results = [result_t1, result_t2, result_t3]

rows = []
for r in all_results:
    analysis = r.get("analysis")
    rows.append({
        "Technique": r["label"],
        "Parse OK": "YES" if r["parse_ok"] else "NO",
        "Valid": "YES" if r["validation_ok"] else "NO",
        "Incident Type": analysis.incident_type if analysis else "—",
        "Severity": analysis.severity.value if analysis else "—",
        "Confidence": f"{analysis.confidence:.3f}" if analysis else "—",
        "# Actions": len(analysis.recommended_actions) if analysis else 0,
        "In Tok": r["input_tokens"],
        "Out Tok": r["output_tokens"],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()

# Compara a consistência geral apenas quando todas as saídas passam na validação.
if all(r["validation_ok"] for r in all_results):
    types = {r["analysis"].incident_type for r in all_results}
    severities = {r["analysis"].severity.value for r in all_results}
    confidences = [r["analysis"].confidence for r in all_results]

    print(f"Incident type agreement : {types}")
    print(f"Severity agreement      : {severities}")
    print(f"Confidence range        : {min(confidences):.3f} – {max(confidences):.3f}")
    print(f"Confidence spread       : {max(confidences) - min(confidences):.3f}")

         Technique Parse OK Valid              Incident Type Severity Confidence  # Actions  In Tok  Out Tok
         T1: RCTOF      YES   YES Data Encryption Ransomware     high      0.900          2     382      102
    T2: Guided CoT      YES    NO                          —        —          —          0     602      150
T3: Meta Prompting       NO    NO                          —        —          —          0     253      501



---
## 10. Discussão

### Técnica 1 — RCTOF

**Pontos fortes:**
- Overhead mínimo de tokens, com uma única geração e prompt curto
- Estrutura previsível de saída, pois o esquema é imposto no próprio prompt
- Boa aderência a pipelines de classificação em alto volume e sensíveis a latência

**Limitações:**
- Não oferece trilha explícita de raciocínio para auditoria
- Pode falhar silenciosamente em incidentes ambíguos, com confiança inflada
- Aderência ao schema depende do wording do prompt e do comportamento do modelo

---

### Técnica 2 — Guided Chain of Thought

**Pontos fortes:**
- Tendência a melhor calibração de confiança, já que o prompt força avaliação baseada em evidências
- Melhores notas de severidade, pois a tríade CIA fornece uma base mais disciplinada
- Lida melhor com incidentes ambíguos, fazendo a incerteza aparecer com mais naturalidade

**Limitações:**
- Custo maior em tokens por causa do prompt mais longo
- Latência maior em comparação com a técnica RCTOF
- Pode ser excessiva para incidentes muito claros e diretos

---

### Técnica 3 — Meta Prompting

**Pontos fortes:**
- Revela estruturas de prompt que o engenheiro talvez não tivesse considerado
- Útil como ferramenta de descoberta para novas categorias de incidente ou novos domínios
- Permite reexecutar a etapa meta quando os tipos de incidente ou o contexto organizacional mudarem

**Limitações:**
- A geração do prompt é não determinística e pode variar entre execuções
- Exige duas gerações, aumentando custo computacional e tempo total
- O prompt gerado ainda precisa de revisão humana antes de ir para produção
- Pode introduzir restrições alucinadas ou enquadramentos subótimos

---

### Validação de Schema com Pydantic

O uso de Pydantic fornece:
- **Mensagens de erro por campo** para depurar respostas malformadas
- **Enforcement de Enum** em `severity`, bloqueando strings inválidas antes de propagarem
- **Validação de intervalo** em `confidence`, evitando floats fora da faixa esperada
- Um contrato de schema reutilizável e versionável, independente da técnica de prompting

A cadeia de fallback em `parse_json_response` (direto → remoção de fences → regex gulosa) cobre os modos de falha mais comuns sem sacrificar precisão.

---
## 11. Conclusão

Este notebook demonstrou três técnicas de prompt engineering aplicadas à triagem de incidentes de cibersegurança com um **modelo open source do Hugging Face**:

| Técnica | Melhor encaixe | Compromisso principal |
|---------|----------------|-----------------------|
| **RCTOF** | Pipelines de alto volume, incidentes claros | Rápida e simples, mas sem trilha de raciocínio |
| **Guided CoT** | Incidentes ambíguos, decisões sensíveis à calibração | Mais consistente, porém com prompts mais longos |
| **Meta Prompting** | Descoberta de prompts, novas categorias | Estrutura nova, porém com variabilidade entre execuções |

### Principais conclusões

1. **A estrutura do prompt importa mais do que o tamanho.** Um schema bem especificado reduz a maior parte das falhas de parsing de JSON, independentemente da técnica.

2. **O raciocínio guiado tende a melhorar a calibração.** As notas de confiança e severidade ficam mais ancoradas nas evidências quando o prompt força uma passagem disciplinada pela tríade CIA.

3. **A validação com Pydantic é uma barreira rígida.** Sistemas SOC a jusante não devem processar saídas de LLM sem validação prévia; o schema é o contrato entre o modelo e o pipeline.

4. **Meta prompting é ferramenta de pesquisa, não atalho de produção.** Prompts gerados precisam de revisão humana e versionamento como qualquer outro artefato de engenharia.